# FLP-NAR Colab Workflow (Refactored)

This notebook mirrors the workflow from `test.ipynb`, but uses refactored functions from the `src` package:
- dataset generation for training/validation from `src.training.training`
- training loop via `src.training.training.Trainer`
- test dataset generation and inference from `src.evaluation.evaluation`

It is structured to run on Google Colab after cloning the repository.

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

In [ ]:
# Resolve repository root so `src` imports work both locally and on Colab.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / "FLP-NAR", Path("/content/FLP-NAR")]
repo_root = None
for c in candidates:
    if (c / "src").exists():
        repo_root = c
        break

if repo_root is None:
    raise RuntimeError("Could not find repository root containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

print(f"Using repo root: {repo_root}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Optional: install dependencies in a fresh Colab runtime.
# Uncomment if needed.
# !pip install -q torch torchvision torchaudio
# !pip install -q torch-geometric
# !pip install -q pandas matplotlib tqdm

In [ ]:
from src.data.data import LossConfig
from src.models.reasoners import Reasoner
from src.training.training import Trainer, generate_training_validation_datasets
from src.evaluation.evaluation import create_test_datasets, run_inference

In [ ]:
# Reproducibility
SEED = 33
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Dataset configs (mirrors test.ipynb scales)
TRAIN_CONFIGS = [
    {"n_fac": 10, "n_cli": 5, "n_samples": 500},
    {"n_fac": 5, "n_cli": 10, "n_samples": 500},
    {"n_fac": 10, "n_cli": 10, "n_samples": 2000},
]

VAL_CONFIGS = [
    {"n_fac": 10, "n_cli": 10, "n_samples": 200},
    {"n_fac": 15, "n_cli": 15, "n_samples": 100},
    {"n_fac": 20, "n_cli": 20, "n_samples": 50},
    {"n_fac": 30, "n_cli": 30, "n_samples": 30},
    {"n_fac": 100, "n_cli": 100, "n_samples": 15},
]

BATCH_SIZE = 256
train_loaders, val_loaders_by_scale = generate_training_validation_datasets(
    train_configs=TRAIN_CONFIGS,
    val_configs=VAL_CONFIGS,
    batch_size=BATCH_SIZE,
)

print(f"train loaders: {len(train_loaders)}")
print(f"val scales: {list(val_loaders_by_scale.keys())}")

In [ ]:
# Model + trainer setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Reasoner(hidden_dim=128, tf_prob=0.5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_config = LossConfig()
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-7,
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss_config=loss_config,
    lr_scheduler=lr_scheduler,
    device=device,
    checkpoint_dir="checkpoints",
)

print(f"device: {device}")
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train (set small epochs for quick Colab smoke test, then scale up)
N_EPOCHS = 10
history = trainer.fit(
    train_loaders=train_loaders,
    val_loaders=val_loaders_by_scale,
    n_epochs=N_EPOCHS,
    verbose=True,
)

In [ ]:
# Build test datasets at multiple scales
TEST_SIZES = [
    (5, 5),
    (10, 10),
    (15, 15),
    (20, 20),
    (10, 20),
    (20, 10),
    (30, 30),
    (50, 50),
    (100, 100),
    (200, 200),
]

# `type` can be: "metric", "weighted", "random"
test_datasets = create_test_datasets(
    sizes=TEST_SIZES,
    n_samples=20,
    type="metric",
    exact=True,
    base_seed=9999,
)

print(f"test sizes: {len(test_datasets)}")

In [ ]:
# If you want to evaluate the best checkpoint from training
trainer.load_checkpoint("checkpoints/best_model.pt")

results_df = run_inference(
    datasets=test_datasets,
    trainer=trainer,
    repair=True,
)

results_df.head()

In [ ]:
# Summary table (mirrors test.ipynb metrics)
summary = results_df.groupby("size").agg(
    opt_ratio_mean=("opt_ratio", "mean"),
    opt_ratio_std=("opt_ratio", "std"),
    gap_pct_mean=("opt_gap_pct", "mean"),
    gap_pct_median=("opt_gap_pct", "median"),
    fac_opened_mean=("n_fac_opened", "mean"),
    fac_target_mean=("n_fac_target", "mean"),
    count=("opt_ratio", "count"),
).round(4)
summary

In [ ]:
# Basic visualization suite
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Optimality gap by size
results_df.boxplot(column="opt_gap_pct", by="size", ax=axes[0, 0])
axes[0, 0].set_title("Optimality Gap (%) by Size")
axes[0, 0].set_ylabel("Gap %")
axes[0, 0].axhline(0, color="g", ls="--", alpha=0.5)

# Predicted vs optimum
axes[0, 1].scatter(results_df["optimum"], results_df["predicted"], alpha=0.6)
mx = max(results_df["optimum"].max(), results_df["predicted"].max())
axes[0, 1].plot([0, mx], [0, mx], "k--", alpha=0.5)
axes[0, 1].set_xlabel("Optimum")
axes[0, 1].set_ylabel("Predicted")
axes[0, 1].set_title("Predicted vs Optimum")

# Opt ratio histogram
axes[1, 0].hist(results_df["opt_ratio"], bins=30, alpha=0.75)
axes[1, 0].axvline(1.0, color="g", ls="--", alpha=0.5)
axes[1, 0].set_title("Opt Ratio Distribution")
axes[1, 0].set_xlabel("opt_ratio")

# Predicted vs dual bound
axes[1, 1].scatter(results_df["dual_bound"], results_df["predicted"], alpha=0.6)
mx2 = max(results_df["dual_bound"].max(), results_df["predicted"].max())
axes[1, 1].plot([0, mx2], [0, mx2], "k--", alpha=0.5)
axes[1, 1].set_xlabel("Dual Bound")
axes[1, 1].set_ylabel("Predicted")
axes[1, 1].set_title("Predicted vs Dual Bound")

plt.suptitle("")
plt.tight_layout()
plt.show()